In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LinearRegression

TISSUE_NAMES = [
    "Tumor", "Empty", "Fibrous", "Inflammation",
    "Necrosis", "Normal", "Reactive", "Steatosis"
]

In [ ]:
def spearman_screen(
        df,
        feature_cols,
        risk_col="risk",
        min_nonzero_frac=0.05,
        min_n=30,
        fdr_method="fdr_bh",
):
    results = []

    for col in feature_cols:
        if col not in df.columns:
            continue

        tmp = df[[col, risk_col]].replace([np.inf, -np.inf], np.nan).dropna()

        if len(tmp) < min_n:
            continue

        if tmp[col].nunique() <= 1:
            continue

        nonzero_frac = (tmp[col] != 0).mean()
        if nonzero_frac < min_nonzero_frac:
            continue

        rho, p = spearmanr(tmp[col], tmp[risk_col])

        results.append({
            "feature": col,
            "spearman_rho": rho,
            "p_value": p,
            "n": len(tmp),
            "nonzero_frac": nonzero_frac,
            "mean": tmp[col].mean(),
            "std": tmp[col].std(),
        })

    res = pd.DataFrame(results)

    if len(res) > 0:
        res["p_adj_BH"] = multipletests(
            res["p_value"],
            method=fdr_method
        )[1]
        res = res.sort_values("p_value")

    return res


def add_delta_and_enrichment_features(df, tissue_names=TISSUE_NAMES, eps=1e-6):
    df = df.copy()
    for tissue in tissue_names:
        sub_col = f"subgraph_{tissue}_pixel_ratio"
        wsi_col = f"wsi_{tissue}_pixel_ratio"

        if sub_col in df.columns and wsi_col in df.columns:
            df[f"{tissue}_delta"] = df[sub_col] - df[wsi_col]
            df[f"{tissue}_enrichment"] = df[sub_col] / (df[wsi_col] + eps)

    return df


def get_feature_groups(df, tissue_names=TISSUE_NAMES):
    # 1. patch-level
    patch_ratio_cols = [
        f"{tissue}_ratio"
        for tissue in tissue_names
        if f"{tissue}_ratio" in df.columns
    ]

    # 2.pixel-level
    subgraph_pixel_cols = [
        f"subgraph_{tissue}_pixel_ratio"
        for tissue in tissue_names
        if f"subgraph_{tissue}_pixel_ratio" in df.columns
    ]

    # 3. WSI pixel-level
    wsi_pixel_cols = [
        f"wsi_{tissue}_pixel_ratio"
        for tissue in tissue_names
        if f"wsi_{tissue}_pixel_ratio" in df.columns
    ]

    # 4. delta
    delta_cols = [
        f"{tissue}_delta"
        for tissue in tissue_names
        if f"{tissue}_delta" in df.columns
    ]

    # 5. enrichment
    enrichment_cols = [
        f"{tissue}_enrichment"
        for tissue in tissue_names
        if f"{tissue}_enrichment" in df.columns
    ]

    # 6. edge
    edge_ratio_cols = [
        c for c in df.columns
        if c.endswith("_edge_ratio")
        and not c.endswith("_edge_weight_ratio")
    ]

    # 7. edge weight
    edge_weight_ratio_cols = [
        c for c in df.columns
        if c.endswith("_edge_weight_ratio")
    ]

    return {
        "01_subgraph_patch_ratio_vs_risk": patch_ratio_cols,
        "02_subgraph_pixel_ratio_vs_risk": subgraph_pixel_cols,
        "03_wsi_pixel_ratio_vs_risk": wsi_pixel_cols,
        "04_subgraph_minus_wsi_delta_vs_risk": delta_cols,
        "05_subgraph_over_wsi_enrichment_vs_risk": enrichment_cols,
        "06_edge_ratio_vs_risk": edge_ratio_cols,
        "07_edge_weight_ratio_vs_risk": edge_weight_ratio_cols,
    }


def run_pgexplainer_pathfinder_risk_correlation(
        csv_path,
        risk_col="risk",
        save_prefix="pgexplainer_pathfinder_risk",
        tissue_names=TISSUE_NAMES,
        min_nonzero_frac=0.05,
        min_n=30,
):

    df = pd.read_csv(csv_path)

    if risk_col not in df.columns:
        raise ValueError(
            f"Risk column '{risk_col}' not found. "
            f"Available columns include: {list(df.columns[:20])} ..."
        )

    df = add_delta_and_enrichment_features(df, tissue_names=tissue_names)
    feature_groups = get_feature_groups(df, tissue_names=tissue_names)

    all_results = {}

    for group_name, cols in feature_groups.items():
        print(f"\n=== Running {group_name} ===")
        print(f"Number of features: {len(cols)}")

        res = spearman_screen(
            df=df,
            feature_cols=cols,
            risk_col=risk_col,
            min_nonzero_frac=min_nonzero_frac,
            min_n=min_n,
        )

        all_results[group_name] = res

        save_path = f"{save_prefix}_{group_name}.csv"
        res.to_csv(save_path, index=False)

        print(res.head(10))
        print(f"[Saved] {save_path}")

    full_save_path = f"{save_prefix}_full_features_with_delta_enrichment.csv"
    df.to_csv(full_save_path, index=False)
    print(f"\n[Saved full feature table] {full_save_path}")

    return df, all_results

In [ ]:
df_full, all_results = run_pgexplainer_pathfinder_risk_correlation(
    csv_path="/root/Desktop/data/private/hjx_product/results/H2GCN_spatial/tissue_summary.csv",
    risk_col="risk",
    save_prefix="/root/Desktop/data/private/hjx_product/results/H2GCN_spatial/statistic_analysis/",
    min_nonzero_frac=0.05,
    min_n=30,
)

In [ ]:
def spearman_screen(
        df,
        feature_cols,
        risk_col="risk",
        min_nonzero_frac=0.05,
        min_n=30,
        fdr_method="fdr_bh",
):
    results = []

    for col in feature_cols:
        if col not in df.columns:
            continue

        tmp = df[[col, risk_col]].replace([np.inf, -np.inf], np.nan).dropna()

        if len(tmp) < min_n:
            continue

        if tmp[col].nunique() <= 1:
            continue

        nonzero_frac = (tmp[col] != 0).mean()
        if nonzero_frac < min_nonzero_frac:
            continue

        rho, p = spearmanr(tmp[col], tmp[risk_col])

        results.append({
            "feature": col,
            "spearman_rho": rho,
            "p_value": p,
            "n": len(tmp),
            "nonzero_frac": nonzero_frac,
            "mean": tmp[col].mean(),
            "std": tmp[col].std(),
        })

    res = pd.DataFrame(results)

    if len(res) > 0:
        res["p_adj_BH"] = multipletests(res["p_value"], method=fdr_method)[1]
        res = res.sort_values("p_value")

    return res


def add_delta_and_enrichment_features(
        df,
        tissue_names=TISSUE_NAMES,
        eps=1e-6,
):
    df = df.copy()

    for tissue in tissue_names:
        sub_col = f"topk_{tissue}_pixel_ratio"
        wsi_col = f"wsi_{tissue}_pixel_ratio"

        if sub_col in df.columns and wsi_col in df.columns:
            df[f"{tissue}_delta"] = df[sub_col] - df[wsi_col]
            df[f"{tissue}_enrichment"] = df[sub_col] / (df[wsi_col] + eps)

    return df


def get_feature_groups(df, tissue_names=TISSUE_NAMES):

    patch_ratio_cols = [
        f"topk_{tissue}_patch_ratio"
        for tissue in tissue_names
        if f"topk_{tissue}_patch_ratio" in df.columns
    ]

    subgraph_pixel_cols = [
        f"topk_{tissue}_pixel_ratio"
        for tissue in tissue_names
        if f"topk_{tissue}_pixel_ratio" in df.columns
    ]

    wsi_pixel_cols = [
        f"wsi_{tissue}_pixel_ratio"
        for tissue in tissue_names
        if f"wsi_{tissue}_pixel_ratio" in df.columns
    ]

    delta_cols = [
        f"{tissue}_delta"
        for tissue in tissue_names
        if f"{tissue}_delta" in df.columns
    ]

    enrichment_cols = [
        f"{tissue}_enrichment"
        for tissue in tissue_names
        if f"{tissue}_enrichment" in df.columns
    ]

    return {
        "01_subgraph_patch_ratio_vs_risk": patch_ratio_cols,
        "02_subgraph_pixel_ratio_vs_risk": subgraph_pixel_cols,
        "03_wsi_pixel_ratio_vs_risk": wsi_pixel_cols,
        "04_subgraph_minus_wsi_delta_vs_risk": delta_cols,
        "05_subgraph_over_wsi_enrichment_vs_risk": enrichment_cols,
    }


def run_pgexplainer_pathfinder_risk_correlation(
        csv_path,
        risk_col="risk",
        save_prefix="pgexplainer_pathfinder_risk",
        tissue_names=TISSUE_NAMES,
        min_nonzero_frac=0.05,
        min_n=30,
):
    df = pd.read_csv(csv_path)

    if risk_col not in df.columns:
        raise ValueError(
            f"Risk column '{risk_col}' not found. "
            f"Available columns include: {list(df.columns[:20])} ..."
        )

    df = add_delta_and_enrichment_features(df, tissue_names=tissue_names)
    feature_groups = get_feature_groups(df, tissue_names=tissue_names)

    all_results = {}

    for group_name, cols in feature_groups.items():
        print(f"\n=== Running {group_name} ===")
        print(f"Number of features: {len(cols)}")

        res = spearman_screen(
            df=df,
            feature_cols=cols,
            risk_col=risk_col,
            min_nonzero_frac=min_nonzero_frac,
            min_n=min_n,
        )

        all_results[group_name] = res

        save_path = f"{save_prefix}_{group_name}.csv"
        res.to_csv(save_path, index=False)

        print(res.head(10))
        print(f"[Saved] {save_path}")

    full_save_path = f"{save_prefix}_full_features_with_delta_enrichment.csv"
    df.to_csv(full_save_path, index=False)
    print(f"\n[Saved full feature table] {full_save_path}")

    return df, all_results

In [ ]:
df_full, all_results = run_pgexplainer_pathfinder_risk_correlation(
    csv_path="/root/Desktop/data/private/hjx_product/results/AMIL/tissue_summary.csv",
    risk_col="risk",
    save_prefix="/root/Desktop/data/private/hjx_product/results/AMIL/statistic_analysis/",
    min_nonzero_frac=0.05,
    min_n=30,
)